# Building GPT with Hardware-Adaptive Optimizations

This notebook implements a GPT language model that **automatically optimizes for your hardware**, whether you're on NVIDIA datacenter GPUs (A100, H100), Apple Silicon (M1/M2/M3/M4), or CPU.

## 🔄 Interruption-Resilient Training

**Perfect for Lightning Studio and cloud environments!** This notebook automatically saves checkpoints and can resume training seamlessly if interrupted:
- ✅ Automatic checkpoint detection on restart
- ✅ Resume from exact training step
- ✅ Preserve optimizer state and learning rate
- ✅ Continue same W&B run for unified metrics
- ✅ Configurable save frequency and cleanup
- ✅ Early stopping to prevent overfitting

Simply re-run the notebook cells if interrupted - it will automatically pick up where it left off!

## What You'll Learn

1. **Character-level language modeling** with transformer architecture
2. **Multi-head attention** and how it enables learning long-range dependencies
3. **Hardware-specific optimizations** for maximum performance
4. **Mixed precision training** (BF16 on A100, FP16 on Apple Silicon)
5. **Memory-efficient training** strategies
6. **Production-grade checkpointing** for resilient training workflows
7. **Early stopping** to prevent overfitting and save compute

## Hardware Optimizations

This notebook automatically detects your hardware and applies optimal settings:

**NVIDIA GPUs (A100, H100, etc.):**
- Mixed precision training (BF16) for 2-3x speedup
- Flash Attention 2 for 2-4x faster attention (if available)
- torch.compile for additional 1.5-2x speedup
- Fused AdamW optimizer
- Larger batch sizes (256) to utilize high memory
- TF32 acceleration on Ampere+ GPUs

**Apple Silicon (M1/M2/M3/M4):**
- MPS (Metal) backend for GPU acceleration
- Mixed precision (FP16) for 1.5-2x speedup
- Memory-efficient batched attention
- Adaptive batch sizing based on available memory
- Gradient accumulation for effective larger batches
- Unified memory optimizations

**CPU Fallback:**
- Smaller batch sizes
- Full FP32 precision
- Memory-efficient operations

## Configuration

All hyperparameters in one place. These will be auto-adjusted based on detected hardware.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 1337,
    
    # Data (will be adjusted based on hardware)
    'batch_size': 64,  # Default, will be optimized per-device
    'block_size': 256,  # Context length
    
    # Model architecture
    'n_embed': 384,  # Embedding dimension
    'n_layers': 6,  # Number of transformer blocks
    'n_heads': 6,  # Number of attention heads
    'dropout': 0.2,  # Dropout rate
    
    # Training
    'learning_rate': 3e-4,
    'max_steps': 5000,
    'eval_interval': 100,
    'eval_iters': 200,
    
    # Early Stopping
    'early_stopping_enabled': True,  # Enable early stopping
    'early_stopping_patience': 5,  # Stop after N validations without improvement
    'early_stopping_min_delta': 0.005,  # Minimum change to count as improvement
    
    # Checkpointing & Resumption
    'checkpoint_dir': 'checkpoints',  # Directory for checkpoints
    'save_every_n_steps': 500,  # Save checkpoint every N steps
    'keep_last_n_checkpoints': 3,  # Keep only last N checkpoints (disk space)
    
    # Hardware-specific (auto-configured)
    'use_mixed_precision': True,  # Enable mixed precision if supported
    'use_flash_attention': True,  # Use Flash Attention 2 if available (CUDA only)
    'use_compile': True,  # Use torch.compile if supported
    'use_fused_optimizer': True,  # Use fused AdamW (CUDA only)
    'use_gradient_accumulation': False,  # Enable for low-memory devices
    'gradient_accumulation_steps': 4,  # Steps to accumulate gradients
}

## Checkpoint Detection for Resumption

Detect if there's an existing checkpoint to resume from. This allows training to continue seamlessly after interruptions.

In [ ]:
import os
import glob
from pathlib import Path

def find_latest_checkpoint(checkpoint_dir):
    """Find the most recent checkpoint in the checkpoint directory."""
    checkpoint_path = Path(checkpoint_dir)
    if not checkpoint_path.exists():
        return None
    
    # Look for Lightning checkpoint files
    checkpoints = list(checkpoint_path.glob('**/*.ckpt'))
    if not checkpoints:
        return None
    
    # Sort by modification time (most recent first)
    checkpoints.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return str(checkpoints[0])

def find_wandb_run_id(checkpoint_dir):
    """Find saved W&B run ID if it exists."""
    run_id_file = Path(checkpoint_dir) / 'wandb_run_id.txt'
    if run_id_file.exists():
        return run_id_file.read_text().strip()
    return None

def save_wandb_run_id(checkpoint_dir, run_id):
    """Save W&B run ID for resumption."""
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    run_id_file = checkpoint_path / 'wandb_run_id.txt'
    run_id_file.write_text(run_id)

# Check for existing checkpoint
CHECKPOINT_PATH = find_latest_checkpoint(CONFIG['checkpoint_dir'])
WANDB_RUN_ID = find_wandb_run_id(CONFIG['checkpoint_dir'])

if CHECKPOINT_PATH:
    print("🔄 RESUMING FROM CHECKPOINT", flush=True)
    print("=" * 70, flush=True)
    print(f"  Checkpoint: {CHECKPOINT_PATH}", flush=True)
    print(f"  Last modified: {Path(CHECKPOINT_PATH).stat().st_mtime}")
    if WANDB_RUN_ID:
        print(f"  W&B Run ID: {WANDB_RUN_ID}", flush=True)
    print("=" * 70, flush=True)
    print("\n✓ Training will continue from the last saved state\n", flush=True)
else:
    print("🆕 STARTING NEW TRAINING", flush=True)
    print("=" * 70, flush=True)
    print(f"  Checkpoints will be saved to: {CONFIG['checkpoint_dir']}/", flush=True)
    print(f"  Save frequency: every {CONFIG['save_every_n_steps']} steps", flush=True)
    print(f"  Keeping last {CONFIG['keep_last_n_checkpoints']} checkpoints", flush=True)
    print("=" * 70, flush=True)
    print("\n✓ New training session will begin\n", flush=True)

## Hardware Detection and Auto-Configuration

Detect the available hardware and automatically configure optimal settings.

In [ ]:
import torch
from aiml_notebooks import set_seed, detect_hardware

set_seed(CONFIG['seed'])

# Detect hardware and configure optimal settings
hw_config = detect_hardware(base_batch_size=CONFIG['batch_size'])

# Update CONFIG with hardware-specific settings
CONFIG['batch_size'] = hw_config.batch_size
CONFIG['use_mixed_precision'] = hw_config.precision != '32-true'
CONFIG['use_flash_attention'] = hw_config.use_flash_attention
CONFIG['use_compile'] = hw_config.use_compile
CONFIG['use_fused_optimizer'] = hw_config.use_fused_optimizer
if hw_config.gradient_accumulation_steps > 1:
    CONFIG['use_gradient_accumulation'] = True
    CONFIG['gradient_accumulation_steps'] = hw_config.gradient_accumulation_steps

# Store hardware config for later use
DEVICE_TYPE = hw_config.device_type
device = hw_config.device
precision_type = hw_config.precision
pin_memory = hw_config.pin_memory

## Load Tiny Shakespeare Dataset

We'll train on Shakespeare's works - a classic character-level language modeling task.

In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(f"Dataset length: {len(text):,} characters", flush=True)
print(f"\nFirst 200 characters:", flush=True)
print(text[:200], flush=True)

## Build Character-Level Tokenizer

Create a simple vocabulary mapping each unique character to an integer.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}", flush=True)
print(f"Characters: {''.join(chars[:20])}...", flush=True)
print(f"\nTest encoding:", flush=True)
test_text = "Hello"
encoded = encode(test_text)
print(f"  '{test_text}' → {encoded}", flush=True)
print(f"  {encoded} → '{decode(encoded)}'", flush=True)

## Create Train/Validation Split

Split into 90% training and 10% validation sets.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}", flush=True)
print(f"Validation tokens: {len(val_data):,}", flush=True)
print(f"Train/val split: {len(train_data)/len(data)*100:.1f}% / {len(val_data)/len(data)*100:.1f}%", flush=True)

## Lightning DataModule

Create a PyTorch Lightning DataModule with hardware-optimized data loading settings.

**DataLoader Settings:**
- **num_workers=0**: In-memory data doesn't need parallel loading
- **pin_memory**: True for CUDA (faster CPU→GPU transfers), False for unified memory (MPS/CPU)
- **persistent_workers**: Not needed when num_workers=0

In [ ]:
import lightning as L
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """Character-level dataset that returns random sequences."""
    
    def __init__(self, data, block_size, num_samples):
        self.data = data
        self.block_size = block_size
        self.num_samples = num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        i = torch.randint(len(self.data) - self.block_size, (1,)).item()
        x = self.data[i:i+self.block_size]
        y = self.data[i+1:i+self.block_size+1]
        return x, y

class ShakespeareDataModule(L.LightningDataModule):
    """DataModule for Shakespeare character-level data."""
    
    def __init__(self, train_data, val_data, batch_size, block_size, eval_iters, pin_memory, max_steps, eval_interval):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = batch_size
        self.block_size = block_size
        self.eval_iters = eval_iters
        self.pin_memory = pin_memory
        
        # Calculate samples per epoch to align with validation intervals
        # This ensures validation happens at reasonable intervals
        self.samples_per_epoch = max_steps // eval_interval * eval_interval * batch_size
    
    def train_dataloader(self):
        # Create dataset sized to validation intervals for clean epoch boundaries
        dataset = CharDataset(self.train_data, self.block_size, num_samples=self.samples_per_epoch)
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            num_workers=0,
            pin_memory=self.pin_memory,
            persistent_workers=False
        )
    
    def val_dataloader(self):
        dataset = CharDataset(self.val_data, self.block_size, 
                            num_samples=self.eval_iters * self.batch_size)
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            num_workers=0,
            pin_memory=self.pin_memory,
            persistent_workers=False
        )

datamodule = ShakespeareDataModule(
    train_data=train_data,
    val_data=val_data,
    batch_size=CONFIG['batch_size'],
    block_size=CONFIG['block_size'],
    eval_iters=CONFIG['eval_iters'],
    pin_memory=pin_memory,
    max_steps=CONFIG['max_steps'],
    eval_interval=CONFIG['eval_interval']
)

# Calculate actual batches per epoch
batches_per_epoch = datamodule.samples_per_epoch // CONFIG['batch_size']
epochs_to_complete = CONFIG['max_steps'] / batches_per_epoch

print(f"DataModule created", flush=True)
print(f"  Batch size: {CONFIG['batch_size']}", flush=True)
print(f"  Block size: {CONFIG['block_size']}", flush=True)
print(f"  Pin memory: {pin_memory}", flush=True)
print(f"  Samples per epoch: {datamodule.samples_per_epoch:,}", flush=True)
print(f"  Batches per epoch: {batches_per_epoch:,}", flush=True)
print(f"  Training steps: {CONFIG['max_steps']:,}", flush=True)
print(f"  Epochs to complete: {epochs_to_complete:.1f}", flush=True)
print(f"  Validation interval: every {CONFIG['eval_interval']} steps", flush=True)

## Check Flash Attention Availability

Flash Attention 2 provides 2-4x speedup on CUDA GPUs. If not installed, we'll use optimized standard attention.

In [ ]:
from aiml_notebooks import check_flash_attention

FLASH_AVAILABLE = check_flash_attention()

if CONFIG['use_flash_attention'] and DEVICE_TYPE == 'cuda':
    if FLASH_AVAILABLE:
        print("✓ Flash Attention 2 is available", flush=True)
    else:
        print("ℹ Flash Attention 2 not available", flush=True)
        print("  Install with: pip install flash-attn", flush=True)
        print("  Will use optimized standard attention instead", flush=True)
        CONFIG['use_flash_attention'] = False
else:
    print(f"Flash Attention: Not applicable for {DEVICE_TYPE}", flush=True)

## Multi-Head Attention Layer

Implement attention with two backends:
1. **Flash Attention**: Used on CUDA if available (2-4x faster, less memory)
2. **Optimized Standard**: Batched attention for all other platforms

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from aiml_notebooks import flash_attention_func

class MultiHeadAttention(nn.Module):
    """Multi-head attention with hardware-optimized backends."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        assert n_embed % n_heads == 0
        
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads
        self.n_embed = n_embed
        self.use_flash = use_flash and FLASH_AVAILABLE
        
        # Combined QKV projection (more efficient)
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.proj = nn.Linear(n_embed, n_embed)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Causal mask (only for standard attention)
        if not self.use_flash:
            self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Combined QKV projection
        qkv = self.qkv(x)  # (B, T, 3*n_embed)
        q, k, v = qkv.chunk(3, dim=-1)  # Each is (B, T, n_embed)
        
        # Reshape to (B, T, n_heads, head_size)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        
        if self.use_flash:
            # Use unified flash attention interface (auto-detects Flash Attention 2)
            out = flash_attention_func(
                q, k, v,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                causal=True
            )  # Returns (B, T, n_heads, head_size)
            out = out.contiguous().view(B, T, self.n_embed)
        else:
            # Standard scaled dot-product attention (batched)
            q = q.transpose(1, 2)  # (B, n_heads, T, head_size)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            
            att = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
            att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            out = att @ v  # (B, n_heads, T, head_size)
            out = out.transpose(1, 2).contiguous().view(B, T, self.n_embed)
        
        # Output projection
        out = self.proj_dropout(self.proj(out))
        return out

print(f"Multi-head attention configured", flush=True)
print(f"  Backend: {'Flash Attention 2 (with fallback, flush=True)' if FLASH_AVAILABLE and CONFIG['use_flash_attention'] else 'Optimized standard'}")

## Feed-Forward Network

Standard position-wise feed-forward network with GELU activation.

In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network with GELU activation."""
    
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

## Transformer Block

A complete transformer decoder block: attention → add & norm → feed-forward → add & norm.

In [ ]:
class Block(nn.Module):
    """Transformer decoder block."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        self.sa = MultiHeadAttention(n_embed, n_heads, block_size, dropout, use_flash)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # Attention with residual
        x = x + self.ffwd(self.ln2(x))  # Feed-forward with residual
        return x

## Complete GPT Model

Full GPT model with hardware-adaptive optimizations using PyTorch Lightning.

**Training Features:**
- Automatic perplexity calculation during validation
- Text sample generation at each validation epoch
- Samples logged to W&B for tracking model quality over time

In [ ]:
import time
import wandb

class GPTLanguageModel(L.LightningModule):
    """GPT with automatic hardware optimization."""
    
    def __init__(self, vocab_size, n_embed=CONFIG['n_embed'], 
                 n_layers=CONFIG['n_layers'], n_heads=CONFIG['n_heads'],
                 block_size=CONFIG['block_size'], dropout=CONFIG['dropout'],
                 learning_rate=CONFIG['learning_rate'],
                 use_flash=CONFIG['use_flash_attention']):
        super().__init__()
        self.save_hyperparameters()
        self.block_size = block_size
        
        # Model components
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([
            Block(n_embed, n_heads, block_size, dropout, use_flash) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        # Training time tracking
        self.train_start_time = None
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        
        # Handle gradient accumulation
        if CONFIG['use_gradient_accumulation']:
            loss = loss / CONFIG['gradient_accumulation_steps']
        
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=False)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        
        # Calculate perplexity
        perplexity = torch.exp(loss)
        
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_perplexity', perplexity, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def on_validation_end(self):
        """Generate and log sample text after each validation run."""
        # Generate sample text
        sample_text = self._generate_sample(max_new_tokens=300)
        
        # Log to wandb if available (without explicit step to avoid conflicts)
        if wandb.run is not None:
            wandb.run.log({
                "samples/generated_text": sample_text,
                "samples/html": wandb.Html(f"<pre>{sample_text}</pre>")
            })
        
        # Print sample for visibility
        print(f"\n{'='*80}", flush=True)
        print(f"Generated Sample (Step {self.global_step}, flush=True):")
        print(f"{'='*80}", flush=True)
        print(sample_text, flush=True)
        print(f"{'='*80}\n", flush=True)
    
    def _generate_sample(self, max_new_tokens=300):
        """Generate a sample text from the model."""
        self.eval()
        with torch.no_grad():
            # Start with empty context
            context = torch.zeros((1, 1), dtype=torch.long, device=self.device)
            generated_ids = self.generate(context, max_new_tokens=max_new_tokens)[0].tolist()
            sample_text = decode(generated_ids)
        self.train()
        return sample_text
    
    def on_train_start(self):
        self.train_start_time = time.time()
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        if self.train_start_time is not None:
            elapsed = time.time() - self.train_start_time
            self.log('train_time_seconds', elapsed, prog_bar=False)
            
            # Calculate throughput
            effective_batch = CONFIG['batch_size']
            if CONFIG['use_gradient_accumulation']:
                effective_batch *= CONFIG['gradient_accumulation_steps']
            tokens_processed = (batch_idx + 1) * effective_batch * CONFIG['block_size']
            throughput = tokens_processed / elapsed
            self.log('tokens_per_second', throughput, prog_bar=False)
    
    def on_train_end(self):
        if self.train_start_time is not None:
            total_time = time.time() - self.train_start_time
            print(f"\nTotal training time: {total_time:.2f}s ({total_time/60:.2f} min, flush=True)")
    
    def configure_optimizers(self):
        # Use fused AdamW on CUDA if available
        if CONFIG['use_fused_optimizer'] and DEVICE_TYPE == 'cuda':
            return torch.optim.AdamW(
                self.parameters(), 
                lr=self.hparams.learning_rate,
                fused=True
            )
        else:
            return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
    
    def generate(self, idx, max_new_tokens):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel(vocab_size)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Statistics:", flush=True)
print(f"  Total parameters: {total_params:,}", flush=True)
print(f"  Model size (FP32, flush=True): ~{total_params * 4 / 1e6:.1f} MB")
if CONFIG['use_mixed_precision']:
    print(f"  Model size (FP16/BF16, flush=True): ~{total_params * 2 / 1e6:.1f} MB")

## Apply torch.compile (Optional)

JIT compilation for additional 1.3-1.8x speedup on compatible hardware (works best on CUDA).

**Compilation Mode**: Using `default` mode for fast startup with good performance. `max-autotune` would be ~2x faster but takes 5-10min to compile.

In [ ]:
from aiml_notebooks import apply_torch_compile

if CONFIG['use_compile']:
    model = apply_torch_compile(model, mode='default')
else:
    print("torch.compile disabled for this hardware", flush=True)

## Training Setup

Configure PyTorch Lightning Trainer with hardware-optimized settings and W&B logging.

## Checkpoint Cleanup Callback

Custom callback to automatically clean up old checkpoints and save disk space.

In [ ]:
from lightning.pytorch.callbacks import Callback

class CheckpointCleanupCallback(Callback):
    """Automatically clean up old periodic checkpoints to save disk space."""
    
    def __init__(self, checkpoint_dir, keep_last_n=3):
        super().__init__()
        self.checkpoint_dir = Path(checkpoint_dir)
        self.keep_last_n = keep_last_n
    
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        """Clean up old checkpoints after each batch (checks are cheap)."""
        if not self.checkpoint_dir.exists():
            return
        
        # Find all step-based checkpoints (not best or last)
        step_checkpoints = sorted(
            [f for f in self.checkpoint_dir.glob('step-*.ckpt')],
            key=lambda x: x.stat().st_mtime,
            reverse=True
        )
        
        # Keep only the last N checkpoints (plus last.ckpt and best are separate)
        if len(step_checkpoints) > self.keep_last_n:
            for old_checkpoint in step_checkpoints[self.keep_last_n:]:
                try:
                    old_checkpoint.unlink()
                except Exception:
                    pass  # Ignore errors during cleanup

cleanup_callback = CheckpointCleanupCallback(
    checkpoint_dir=CONFIG['checkpoint_dir'],
    keep_last_n=CONFIG['keep_last_n_checkpoints']
)

print(f"Checkpoint cleanup configured:", flush=True)
print(f"  Will keep last {CONFIG['keep_last_n_checkpoints']} periodic checkpoints", flush=True)
print(f"  Plus best model and last.ckpt (always preserved, flush=True)")

## Cloud Progress Callback

Custom callback for Modal/cloud environments that provides detailed progress logging with explicit stdout flushing.

This ensures progress appears in real-time in Modal logs (Lightning's progress bars don't work in non-TTY environments).

In [ ]:
import sys

class CloudProgressCallback(Callback):
    """Progress callback for Modal/cloud environments with explicit flushing."""

    def __init__(self, log_every_n_steps=10):
        super().__init__()
        self.log_every_n_steps = log_every_n_steps
        self.train_start_step = None

    def on_train_start(self, trainer, pl_module):
        print("\n" + "="*80, flush=True)
        print("🚀 TRAINING STARTED", flush=True)
        print("="*80, flush=True)
        print(f"Device: {pl_module.device}", flush=True)
        print(f"Max steps: {trainer.max_steps}", flush=True)
        print(f"Validation every: {trainer.val_check_interval} steps", flush=True)
        print("="*80 + "\n", flush=True)
        self.train_start_step = trainer.global_step

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if (trainer.global_step + 1) % self.log_every_n_steps == 0:
            loss = outputs['loss'].item() if 'loss' in outputs else 'N/A'
            progress_pct = (trainer.global_step + 1) / trainer.max_steps * 100
            print(
                f"[Step {trainer.global_step + 1:5d}/{trainer.max_steps}] "
                f"({progress_pct:5.1f}%) | Loss: {loss:.4f}",
                flush=True
            )

    def on_validation_start(self, trainer, pl_module):
        print(f"\n{'─'*80}", flush=True)
        print(f"📊 VALIDATION (Step {trainer.global_step})", flush=True)
        print(f"{'─'*80}", flush=True)

    def on_validation_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss', None)
        val_perplexity = trainer.callback_metrics.get('val_perplexity', None)
        print(f"{'─'*80}", flush=True)
        if val_loss is not None:
            print(f"✓ Val Loss: {val_loss:.4f}", flush=True)
        if val_perplexity is not None:
            print(f"✓ Perplexity: {val_perplexity:.2f}", flush=True)
        print(f"{'─'*80}\n", flush=True)

    def on_train_end(self, trainer, pl_module):
        total_steps = trainer.global_step - (self.train_start_step or 0)
        print("\n" + "="*80, flush=True)
        print("🎉 TRAINING COMPLETED", flush=True)
        print("="*80, flush=True)
        print(f"Total steps: {total_steps}", flush=True)
        print("="*80 + "\n", flush=True)

print("CloudProgressCallback loaded for Modal/cloud environments", flush=True)

## Early Stopping

Configure early stopping to prevent overfitting and save compute time.

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping

# Configure early stopping if enabled
early_stopping_callback = None
if CONFIG['early_stopping_enabled']:
    early_stopping_callback = EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['early_stopping_patience'],
        min_delta=CONFIG['early_stopping_min_delta'],
        mode='min',
        verbose=True,
        check_on_train_epoch_end=False  # Check on validation epoch end
    )
    print(f"Early stopping configured:", flush=True)
    print(f"  Monitor: val_loss", flush=True)
    print(f"  Patience: {CONFIG['early_stopping_patience']} validations", flush=True)
    print(f"  Min delta: {CONFIG['early_stopping_min_delta']}", flush=True)
    print(f"  → Training will stop if validation loss doesn't improve", flush=True)
else:
    print("Early stopping disabled - training will run for full max_steps", flush=True)

In [ ]:
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import wandb
import warnings

# Suppress common warnings that are informational only
warnings.filterwarnings('ignore', message='.*num_workers.*')  # DataLoader num_workers (intentionally 0)
warnings.filterwarnings('ignore', message='.*Online softmax.*')  # torch._inductor internal
warnings.filterwarnings('ignore', category=UserWarning, module='torch._dynamo')  # Recompile warnings

# Get Modal run ID for correlation with W&B
modal_run_id = os.environ.get("MODAL_TASK_ID", "local")
modal_app_id = os.environ.get("MODAL_FUNCTION_ID", "unknown")
run_identifier = f"modal-{modal_run_id[:8]}" if modal_run_id != "local" else "local"
print(f"🔗 Run Identifier: {run_identifier}", flush=True)
print(f"📊 Modal Task ID: {modal_run_id}", flush=True)


# Create or resume W&B logger
wandb_logger = None
if WANDB_RUN_ID:
    # Resume existing W&B run by initializing wandb first
    wandb.init(
        project='gpt-shakespeare',
        id=WANDB_RUN_ID,
        resume='must',
        config=CONFIG
    )
    # Create logger with existing run
    wandb_logger = WandbLogger(experiment=wandb.run, log_model=False)
    print(f"Resumed W&B run: {wandb.run.name} (ID: {WANDB_RUN_ID})", flush=True)
else:
    # Create new W&B run (check if already in a sweep first)
    if wandb.run is None:
        wandb_logger = WandbLogger(
            project='gpt-shakespeare',
            name=f'gpt_{DEVICE_TYPE}_{run_identifier}',
            config=CONFIG,
            log_model=False
        )
        # Save run ID for future resumption
        save_wandb_run_id(CONFIG['checkpoint_dir'], wandb_logger.experiment.id)
        print(f"Created W&B run: {wandb_logger.experiment.name}", flush=True)
    else:
        # Already in a sweep, use existing run
        wandb_logger = WandbLogger(experiment=wandb.run, log_model=False)
        print(f"Using existing W&B sweep run: {wandb.run.name}", flush=True)

# Also keep CSV logger for local tracking
csv_logger = CSVLogger('logs', name=f'gpt_{DEVICE_TYPE}_{run_identifier}')

# Use both loggers
loggers = [csv_logger]
if wandb_logger is not None:
    loggers.append(wandb_logger)

# Checkpoint callback for best model (based on validation loss)
best_checkpoint_callback = ModelCheckpoint(
    dirpath=CONFIG['checkpoint_dir'],
    filename='best-{epoch:02d}-{step:06d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=False,
    auto_insert_metric_name=False,
    verbose=False,  # Suppress "checkpoint directory exists" warning
)

# Checkpoint callback for periodic saves (every N steps)
# Using save_top_k=-1 with every_n_train_steps saves all periodic checkpoints
# We'll manually clean up old ones if needed
periodic_checkpoint_callback = ModelCheckpoint(
    dirpath=CONFIG['checkpoint_dir'],
    filename='step-{step:06d}-{epoch:02d}',
    every_n_train_steps=CONFIG['save_every_n_steps'],
    save_top_k=-1,  # Save all when using every_n_train_steps
    save_last=True,  # Always keep the very last checkpoint
    auto_insert_metric_name=False,
    enable_version_counter=False,
    verbose=False,  # Suppress "checkpoint directory exists" warning
)

# Gradient accumulation if enabled
accumulate_grad_batches = 1
if CONFIG['use_gradient_accumulation']:
    accumulate_grad_batches = CONFIG['gradient_accumulation_steps']

# Build callbacks list (includes cloud progress for Modal)
cloud_progress = CloudProgressCallback(log_every_n_steps=10)
callbacks = [
    best_checkpoint_callback,
    periodic_checkpoint_callback,
    cleanup_callback,
    cloud_progress  # Real-time progress for Modal/cloud
]
if early_stopping_callback is not None:
    callbacks.append(early_stopping_callback)

trainer = L.Trainer(
    max_steps=CONFIG['max_steps'],
    val_check_interval=CONFIG['eval_interval'],
    accelerator='auto',
    devices=1,
    precision=precision_type if CONFIG['use_mixed_precision'] else '32-true',
    logger=loggers,
    callbacks=callbacks,
    enable_progress_bar=False,  # Disabled for Modal (progress via CloudProgressCallback)
    log_every_n_steps=1,
    accumulate_grad_batches=accumulate_grad_batches,
    benchmark=(DEVICE_TYPE == 'cuda'),  # cuDNN benchmarking on CUDA
)

effective_batch = CONFIG['batch_size'] * accumulate_grad_batches
print(f"\nTrainer Configuration:", flush=True)
print(f"  Max steps: {CONFIG['max_steps']}", flush=True)
print(f"  Batch size: {CONFIG['batch_size']}", flush=True)
print(f"  Effective batch: {effective_batch}", flush=True)
print(f"  Precision: {precision_type if CONFIG['use_mixed_precision'] else '32-true'}", flush=True)
print(f"  Tokens per step: {effective_batch * CONFIG['block_size']:,}", flush=True)
print(f"\nCheckpointing:", flush=True)
print(f"  Best model: saved to {CONFIG['checkpoint_dir']}/", flush=True)
print(f"  Periodic saves: every {CONFIG['save_every_n_steps']} steps", flush=True)
print(f"  Auto-cleanup: Keeping last {CONFIG['keep_last_n_checkpoints']} periodic checkpoints", flush=True)
if CHECKPOINT_PATH:
    print(f"  Resume from: {CHECKPOINT_PATH}", flush=True)
if early_stopping_callback:
    print(f"\nEarly Stopping: Enabled (patience={CONFIG['early_stopping_patience']}, flush=True)")

## Train the Model

Run training with all hardware-specific optimizations enabled. Training will automatically resume from the last checkpoint if interrupted.

**Checkpoint Strategy:**
- **Best model**: Saved when validation loss improves
- **Periodic saves**: Every N steps (configurable)
- **Last checkpoint**: Always saved for easy resumption
- **Automatic cleanup**: Keeps only last N checkpoints to save disk space

**Early Stopping:**
- Monitors validation loss after each evaluation
- Stops training if loss doesn't improve for N validations (patience)
- Prevents overfitting and saves compute time
- Configurable via `early_stopping_patience` and `early_stopping_min_delta`

**Interruption Recovery:**
If training is interrupted, simply re-run this cell. The notebook will:
1. Detect the most recent checkpoint
2. Resume training from that exact step
3. Continue logging to the same W&B run
4. Preserve all training state (optimizer, learning rate, etc.)
5. Early stopping counter is also restored from checkpoint

In [ ]:
# Print correlation info for Modal ↔ W&B
modal_task_id = os.environ.get("MODAL_TASK_ID", "local")
if modal_task_id != "local":
    print("="*80, flush=True)
    print("🔗 MODAL ↔ W&B CORRELATION", flush=True)
    print("="*80, flush=True)
    print(f"Modal Task ID: {modal_task_id}", flush=True)
    print(f"W&B Run Name: gpt_{DEVICE_TYPE}_modal-{modal_task_id[:8]}", flush=True)
    print(f"View logs: modal logs {modal_task_id}", flush=True)
    print("="*80, flush=True)
    print(flush=True)

if CHECKPOINT_PATH:
    print(f"🔄 Resuming training from checkpoint on {DEVICE_TYPE.upper()}...", flush=True)
    print(f"   Checkpoint: {CHECKPOINT_PATH}", flush=True)
    print(flush=True)
else:
    print(f"🆕 Starting new training on {DEVICE_TYPE.upper()}...", flush=True)
    print(flush=True)

# Train with optional checkpoint resumption
trainer.fit(
    model, 
    datamodule, 
    ckpt_path=CHECKPOINT_PATH  # None for new training, path for resumption
)

## Plot Training Curves and Performance Metrics

Visualize training progression, loss curves, perplexity, and throughput.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(f'{csv_logger.log_dir}/metrics.csv')

train_metrics = metrics[['step', 'train_loss']].dropna()
val_metrics = metrics[['step', 'val_loss']].dropna()
perplexity_metrics = metrics[['step', 'val_perplexity']].dropna()
time_metrics = metrics[['step', 'train_time_seconds']].dropna()
throughput_metrics = metrics[['step', 'tokens_per_second']].dropna()

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])

# Loss curves
ax1.plot(train_metrics['step'], train_metrics['train_loss'],
         label='Train', linewidth=2, alpha=0.7, color='#4ECDC4')
ax1.plot(val_metrics['step'], val_metrics['val_loss'],
         label='Validation', marker='o', linewidth=2, markersize=3, color='#FF6B6B')
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title(f'Training Progress ({DEVICE_TYPE.upper()})', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Perplexity
ax2.plot(perplexity_metrics['step'], perplexity_metrics['val_perplexity'],
         marker='o', linewidth=2, markersize=3, color='#9B59B6')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Perplexity', fontsize=12)
ax2.set_title('Validation Perplexity', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Training time
ax3.plot(time_metrics['step'], time_metrics['train_time_seconds'] / 60,
         linewidth=2, color='#95E1D3')
ax3.set_xlabel('Step', fontsize=12)
ax3.set_ylabel('Elapsed Time (minutes)', fontsize=12)
ax3.set_title('Training Time', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Throughput
ax4.plot(throughput_metrics['step'], throughput_metrics['tokens_per_second'],
         linewidth=2, color='#F38181')
ax4.set_xlabel('Step', fontsize=12)
ax4.set_ylabel('Tokens/Second', fontsize=12)
ax4.set_title('Training Throughput', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# Steps per second
time_metrics['steps_per_sec'] = time_metrics['step'] / time_metrics['train_time_seconds']
ax5.plot(time_metrics['step'], time_metrics['steps_per_sec'],
         linewidth=2, color='#A8E6CF')
ax5.set_xlabel('Step', fontsize=12)
ax5.set_ylabel('Steps/Second', fontsize=12)
ax5.set_title('Training Speed', fontsize=14, fontweight='bold')
ax5.grid(True, alpha=0.3)

plt.show()

# Performance statistics
final_train_loss = train_metrics['train_loss'].iloc[-1]
final_val_loss = val_metrics['val_loss'].iloc[-1]
final_perplexity = perplexity_metrics['val_perplexity'].iloc[-1]
total_time = time_metrics['train_time_seconds'].iloc[-1]
avg_throughput = throughput_metrics['tokens_per_second'].mean()
avg_steps_per_sec = time_metrics['steps_per_sec'].mean()

print(f"\n{'='*70}")
print(f"TRAINING STATISTICS ({DEVICE_TYPE.upper()})")
print(f"{'='*70}")
print(f"\nPerformance:")
print(f"  Total steps: {len(train_metrics):,}")
print(f"  Total time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
print(f"  Average speed: {avg_steps_per_sec:.2f} steps/sec")
print(f"  Average throughput: {avg_throughput:,.0f} tokens/sec")
print(f"  Total tokens: {len(train_metrics) * effective_batch * CONFIG['block_size']:,}")
print(f"\nModel Quality:")
print(f"  Final train loss: {final_train_loss:.4f}")
print(f"  Final val loss: {final_val_loss:.4f}")
print(f"  Final perplexity: {final_perplexity:.2f}")
print(f"\nOptimizations Enabled:")
print(f"  Device: {DEVICE_TYPE.upper()}")
print(f"  Mixed Precision: {'✓' if CONFIG['use_mixed_precision'] else '✗'} ({precision_type if CONFIG['use_mixed_precision'] else 'FP32'})")
print(f"  Flash Attention: {'✓' if FLASH_AVAILABLE and CONFIG['use_flash_attention'] else '✗'}")
print(f"  torch.compile: {'✓' if CONFIG['use_compile'] else '✗'}")
print(f"  Fused Optimizer: {'✓' if CONFIG['use_fused_optimizer'] else '✗'}")
print(f"  Gradient Accumulation: {'✓' if CONFIG['use_gradient_accumulation'] else '✗'}")
print(f"  Batch Size: {CONFIG['batch_size']}")
print(f"  Effective Batch: {effective_batch}")
print(f"{'='*70}")

## Generate Shakespeare-like Text

Use the trained model to generate new text in Shakespeare's style.

In [ ]:
from aiml_notebooks import get_device

device = get_device()

# Handle torch.compile wrapper
if hasattr(model, '_orig_mod'):
    print("Note: First generation may be slow due to torch.compile...")

model = model.to(device)
model.eval()

# Generate from empty context
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print("\nGenerated text:")
print("="*80)
print(generated_text)
print("="*80)

## Key Takeaways

### Production-Grade Training Workflows

This notebook demonstrates **interruption-resilient training**, essential for cloud/studio environments:

**Checkpoint Strategy:**
- **Best model checkpointing**: Automatically saves when validation loss improves
- **Periodic checkpointing**: Regular saves every N steps (default: 500)
- **Last checkpoint**: Always maintains most recent state for instant resumption
- **Automatic cleanup**: Keeps only last N checkpoints to manage disk space
- **W&B run continuity**: Resumes same run for unified metrics tracking

**Early Stopping:**
- **Prevents overfitting**: Stops when validation loss plateaus
- **Saves compute**: No wasted training after convergence
- **Configurable patience**: Wait N validations before stopping (default: 10)
- **Minimum delta**: Only count improvements above threshold (default: 0.001)
- **State preservation**: Early stopping counter restored from checkpoint

**Why This Matters:**
- **Cloud interruptions**: Spot instances, preemption, session timeouts
- **Long training runs**: Multi-hour/day training with peace of mind
- **Experimentation**: Stop/resume training while adjusting hyperparameters
- **Resource efficiency**: No wasted compute from restarts or overfitting

**How to Use:**
1. Training interrupted? Just re-run the notebook
2. Automatic checkpoint detection on restart
3. Training continues from exact step
4. All state preserved (model, optimizer, LR, step count, early stopping)
5. Training stops automatically when model converges

### Hardware-Adaptive Training

This notebook demonstrates how to write **hardware-agnostic** deep learning code that automatically optimizes for different platforms:

**NVIDIA GPUs (A100, H100):**
- **BF16 mixed precision**: 2-3x speedup with better stability than FP16
- **Flash Attention 2**: 2-4x faster attention with reduced memory
- **torch.compile**: Additional 1.5-2x speedup through JIT compilation
- **Fused optimizers**: Single-kernel parameter updates
- **Large batches**: Utilize 40-80GB dedicated memory
- **TF32**: Automatic acceleration on Ampere+ architectures
- **Combined speedup**: 5-8x faster than baseline

**Apple Silicon (M1/M2/M3/M4):**
- **MPS backend**: Native Metal GPU acceleration
- **FP16 mixed precision**: 1.5-2x speedup (BF16 not well-supported)
- **Adaptive batching**: Auto-adjusts based on unified memory (8-128GB)
- **Gradient accumulation**: Simulates larger batches on low-memory devices
- **Unified memory**: No CPU↔GPU transfers needed
- **Combined speedup**: 2-8x faster than CPU (depends on chip)

### Transformer Architecture Insights

**Multi-head attention** is the key innovation:
- Allows model to attend to different aspects of the context
- Each head learns different patterns (syntax, semantics, long-range dependencies)
- Parallel computation across heads enables efficient training
- Causal masking ensures autoregressive property

**Residual connections** enable deep networks:
- Gradient flow through addition operations
- Each block can learn refinements to existing representations
- Allows training 6+ layers without vanishing gradients

**Layer normalization** stabilizes training:
- Pre-norm architecture (norm before attention/FFN) is standard in modern transformers
- Prevents activation explosion in deep networks
- Enables higher learning rates

### Performance Optimization Strategy

**Optimize in this order:**
1. **Early stopping** (prevents overfitting, saves compute)
2. **Checkpointing** (enables long training runs)
3. **Mixed precision** (biggest speedup, easy to enable)
4. **Optimal batch size** (utilize available memory)
5. **Flash Attention** (if on CUDA, 2-4x speedup)
6. **torch.compile** (if PyTorch 2.0+, 1.5-2x speedup)
7. **Gradient accumulation** (if memory-constrained)
8. **Multi-GPU** (if available, linear scaling)

**When to use what:**
- **Prototyping**: Apple Silicon with MPS (convenient, fast enough)
- **Training large models**: NVIDIA datacenter GPUs (A100, H100)
- **Fine-tuning**: Either platform works well
- **Inference**: Quantization + any platform

### Further Exploration

Try these experiments:
1. **Interrupt and resume**: Stop training mid-way, restart notebook, verify seamless resumption
2. **Early stopping tuning**: Adjust patience and min_delta to find optimal stopping point
3. **Checkpoint frequency**: Adjust `save_every_n_steps` for your workflow
4. **Increase model size**: More layers (8-12) or larger embeddings (512, 768)
5. **Longer context**: Increase block_size to 512 or 1024
6. **Better tokenization**: Use BPE (byte-pair encoding) instead of characters
7. **More data**: Train on larger text corpus
8. **Learning rate schedule**: Add warmup and decay
9. **Multi-GPU**: Use DDP for data parallelism

**Key lesson**: Write hardware-agnostic, production-ready code with proper checkpointing and early stopping. The same codebase runs optimally on datacenter GPUs, laptops, workstations, and cloud platforms - and gracefully handles interruptions while preventing overfitting.